# LITE Re-Ranker — Kaggle Training Notebook

This notebook drives the `literank` package (a faithful reference implementation of the
**LITE** document re-ranker, [arXiv:2406.17968](https://arxiv.org/abs/2406.17968)) on
Kaggle's free GPU tier.

**Before running:**

1. In the Kaggle notebook sidebar, set **Accelerator -> GPU T4 x2** and turn **Internet -> On**
   (needed to `pip install`, pull the HuggingFace encoder/teacher models, and download MS MARCO).
2. Kaggle GPU sessions are capped (currently ~30 GPU-hours/week, ~12h/session, often less on a
   shared T4x2). Training to the paper's `max_steps=20000` will usually **span multiple
   sessions**. This notebook checkpoints every `checkpoint_every` steps
   (see `literank.config.TrainConfig`) under `/kaggle/working/ckpt_*`, and the `train` CLI
   command accepts `--resume <path_to_ckpt.pt>` to continue from the last saved step. The final
   markdown cell below explains how to persist `/kaggle/working/ckpt_*` as a Kaggle Dataset so
   the next session can resume training instead of starting over.
3. This is a **qualitative reproduction**, not a leaderboard run: the paper reports MRR@10 =
   0.393 on the full MS MARCO dev set with full-scale training compute. With Kaggle's free GPU
   and a `--subset-size` slice of MS MARCO, expect numbers below 0.393 — the goal is to see the
   LITE scorer learn and to demonstrate the ablations (LITE vs MaxSim, Small-LITE projection
   storage, activation choice), not to match the paper's headline number.


In [ ]:
# Setup: clone the repo and make the `literank` package importable.
# Internet must be ON (sidebar) to clone and to pull HF models + MS MARCO.
import os, sys
REPO = "/kaggle/working/searchandrank"
!test -d {REPO} || git clone -b feat/paper-faithful https://github.com/jaganadhg/searchandrank.git {REPO}
SRC = REPO + "/src"
# Put src/ FIRST on the path for this kernel AND for `!python -m literank.cli` subprocesses.
# (Avoids the fragile editable install and a name clash with the old ./literank draft dir.)
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
%cd {REPO}
# Kaggle ships torch/transformers/datasets/scikit-learn; install only if a dep is missing:
# !pip install -q -U datasets transformers
import literank
print("literank OK ->", literank.__file__)


In [ ]:
# Offline smoke test: fast unit tests only, no network/model downloads.
!pytest -m "not integration" -q
# (If you prefer uv and installed it via `pip install uv`, you can instead run:)
# !uv run pytest -m "not integration" -q


In [ ]:
# Train the LITE scorer on a subset of MS MARCO, distilled from the cross-encoder teacher.
# Checkpoints land in /kaggle/working/ckpt_lite/ckpt_step<N>.pt every `checkpoint_every` steps.
!python -m literank.cli train --scorer lite --proj-dim 768 \
    --subset-size 100000 --max-steps 20000 \
    --checkpoint-dir /kaggle/working/ckpt_lite --device cuda
# Add --eval-every 1000 --patience 3 to stop automatically when dev MRR@10 plateaus


In [ ]:
# Train the MaxSim (ColBERT-style) baseline for the LITE-vs-MaxSim ablation.
!python -m literank.cli train --scorer maxsim --subset-size 100000 \
    --max-steps 20000 --checkpoint-dir /kaggle/working/ckpt_maxsim --device cuda


In [ ]:
# Small-LITE storage ablation: compare cached embedding bytes at proj_dim=768 vs proj_dim=128.
# `encode_and_cache` returns the size (bytes) of the saved cache file -- the "storage lever"
# from the paper's Small-LITE variant (projecting to a smaller d' shrinks the offline doc cache).
import torch
from literank.config import ModelConfig
from literank.encoder import DualEncoder

sample_docs = [
    "The quick brown fox jumps over the lazy dog.",
    "LITE is a lightweight late-interaction re-ranker for document retrieval.",
    "Kaggle provides free GPU compute for training and inference.",
] * 20  # small stand-in batch; the dev-doc cache built below uses the real MS MARCO dev split

from literank.encode_cache import encode_and_cache

cfg_768 = ModelConfig(proj_dim=768)
enc_768 = DualEncoder(cfg_768).to("cuda")
bytes_768 = encode_and_cache(enc_768, sample_docs, cfg_768.doc_len,
                              "/kaggle/working/ablation_proj768.pt", batch_size=32)

cfg_128 = ModelConfig(proj_dim=128)
enc_128 = DualEncoder(cfg_128).to("cuda")
bytes_128 = encode_and_cache(enc_128, sample_docs, cfg_128.doc_len,
                              "/kaggle/working/ablation_proj128.pt", batch_size=32)

print(f"proj_dim=768 cache size: {bytes_768:,} bytes")
print(f"proj_dim=128 cache size: {bytes_128:,} bytes")
print(f"storage ratio (128/768): {bytes_128 / bytes_768:.3f}")
assert bytes_128 < bytes_768, "Small-LITE (proj_dim=128) should use less storage than proj_dim=768"


In [ ]:
# Self-contained eval: finds every trained checkpoint under /kaggle/working, reads each
# checkpoint's own config (works for lite & maxsim, any dir name), and reports MRR@10/nDCG@10.
import glob, os, re, torch
from datasets import load_dataset
from literank.config import ModelConfig, DataConfig
from literank.model import Ranker
from literank.checkpoint import load_checkpoint
from literank.rerank import rerank
from literank.evaluate import mrr_at_k, ndcg_at_k

N_DEV = 2000
SEARCH_DIRS = sorted(glob.glob("/kaggle/working/ckpt_*"))   # add "/kaggle/input/<dataset>/..." dirs if needed

# Build the dev slice once (streaming; keep queries that have at least one relevant passage).
dcfg = DataConfig()
stream = load_dataset(dcfg.dataset_name, dcfg.dataset_config, split="validation", streaming=True)
dev = []
for rec in stream:
    passages = rec["passages"]["passage_text"]
    labels = rec["passages"]["is_selected"]
    if passages and sum(labels) > 0:
        dev.append((rec["query"], passages, labels))
    if len(dev) >= N_DEV:
        break
print(f"dev queries: {len(dev)}")

def pick_checkpoint(d):
    best = os.path.join(d, "best.pt")
    if os.path.exists(best):
        return best
    steps = glob.glob(os.path.join(d, "ckpt_step*.pt"))
    return max(steps, key=lambda p: int(re.search(r"ckpt_step(\d+)\.pt", os.path.basename(p)).group(1))) if steps else None

@torch.no_grad()
def evaluate(ckpt_path):
    blob = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = ModelConfig(**blob["config"])                 # use the checkpoint's own architecture
    ranker = Ranker(cfg).to("cuda")
    load_checkpoint(ckpt_path, ranker, map_location="cuda")
    ranker.eval()
    ranked = []
    for query, passages, labels in dev:
        doc_embs, doc_masks = ranker.encoder.encode(passages, cfg.doc_len)
        q_emb, q_mask = ranker.encoder.encode([query], cfg.query_len)
        order = rerank(ranker.scorer, q_emb, q_mask, doc_embs, doc_masks, device="cuda")
        ranked.append([labels[i] for i in order])
    return cfg.scorer, mrr_at_k(ranked, 10), ndcg_at_k(ranked, 10)

print(f"{'scorer':8} {'dir':26} {'ckpt':16} {'MRR@10':>8} {'nDCG@10':>8}")
found = False
for d in SEARCH_DIRS:
    ck = pick_checkpoint(d)
    if ck is None:
        continue
    found = True
    scorer, mrr, ndcg = evaluate(ck)
    print(f"{scorer:8} {os.path.basename(d):26} {os.path.basename(ck):16} {mrr:8.4f} {ndcg:8.4f}")
if not found:
    print("No checkpoints under /kaggle/working/ckpt_*. If they are in an attached Dataset, "
          "add its /kaggle/input/<name> dir(s) to SEARCH_DIRS above.")
print("\nNote: scores use each query's own candidate passages (not BM25 top-1000); only the "
      "relative LITE-vs-MaxSim comparison is meaningful, not the absolute vs the paper's 0.393.")


## Multi-session training: checkpoint -> Kaggle Dataset -> `--resume`

Kaggle GPU sessions are time-boxed, so a full `--max-steps 20000` run will often need to
continue across multiple sessions. Use the checkpoint files this notebook already writes to
`/kaggle/working/ckpt_lite/` and `/kaggle/working/ckpt_maxsim/` (one `ckpt_step<N>.pt` file per
`checkpoint_every` steps, per `literank.config.TrainConfig`):

1. **At the end of a session**, open the notebook's **Output** tab (or the right-hand "Data"
   pane), and click **"New Dataset"** from the `/kaggle/working/ckpt_lite` (or `ckpt_maxsim`)
   folder. Kaggle will snapshot those checkpoint files as a private Dataset you can re-attach
   to a new session.
   - Alternatively, save the whole notebook ("Save Version") with "Always save output" enabled,
     which persists `/kaggle/working/*` as the notebook's output and lets you create a Dataset
     from it afterward.
2. **In the next session**, add that Dataset as an input (Notebook -> Add Data -> Your
   Datasets), which mounts it read-only under `/kaggle/input/<dataset-name>/`.
3. Copy (or symlink) the most recent `ckpt_step<N>.pt` somewhere writable, then resume.
   Because a session can be interrupted at any step, `<N>` varies with wherever the
   previous session stopped -- it is **not** a fixed value like `ckpt_step20000.pt`.
   Find the checkpoint with the highest `<N>` first, e.g. with the same helper the
   training/eval cells above use:

```python
import glob, os, re

def latest_checkpoint(ckpt_dir):
    paths = glob.glob(os.path.join(ckpt_dir, "ckpt_step*.pt"))
    if not paths:
        raise FileNotFoundError(f"no checkpoints in {ckpt_dir}")
    return max(paths, key=lambda p: int(re.search(r"ckpt_step(\d+)\.pt", os.path.basename(p)).group(1)))

latest_ckpt = latest_checkpoint("/kaggle/input/<dataset-name>")
```

Then copy that file and resume:

```python
!cp <latest_ckpt_path_from_above> /kaggle/working/resume_ckpt.pt
!python -m literank.cli train --scorer lite --proj-dim 768 \
    --subset-size 100000 --max-steps 40000 \
    --checkpoint-dir /kaggle/working/ckpt_lite \
    --resume /kaggle/working/resume_ckpt.pt --device cuda
```

`train --resume <path>` calls `literank.checkpoint.load_checkpoint`, which restores the model,
optimizer, and AMP grad-scaler state and returns the saved `step`, so training continues from
exactly where it left off (raise `--max-steps` past the resumed step to keep going). Repeat the
save-as-Dataset / re-attach / `--resume` loop each session until training reaches the desired
`max_steps`.
